In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1421_Dr._Karni_Singh_Shooting_Range_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,196.21,301.20,10.12,26.68,34.79,40.39,3.09,1.27,11.12,...,NaN,10.89,79.56,0.56,111.18,0.00,0.00,53.05,990.73,NaN
1,2024-01-02,225.53,349.33,20.00,29.37,47.37,38.98,3.54,1.29,12.26,...,NaN,10.21,77.35,0.54,133.05,0.00,0.00,59.52,989.99,NaN
2,2024-01-03,205.20,319.21,21.57,47.43,49.42,40.12,4.69,1.82,10.24,...,NaN,9.83,88.31,0.61,120.46,0.00,0.00,44.92,989.87,NaN
3,2024-01-04,246.69,379.60,20.43,55.09,45.92,36.88,7.32,1.74,3.71,...,NaN,10.23,87.16,1.16,262.06,0.00,0.00,28.61,990.15,NaN
4,2024-01-05,173.72,290.92,18.78,52.80,43.36,41.13,4.93,1.63,11.46,...,NaN,11.06,88.54,0.70,198.65,0.00,0.00,24.23,990.16,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,200.85,249.10,53.74,70.23,79.05,46.10,12.05,1.84,14.23,...,NaN,13.45,86.98,0.97,154.23,0.42,0.42,8.44,991.45,NaN
362,2024-12-28,90.89,120.76,33.28,56.31,55.01,35.44,12.65,1.18,9.74,...,NaN,13.52,90.38,0.73,199.31,0.10,0.10,13.81,990.92,NaN
363,2024-12-29,101.26,133.79,14.91,46.65,34.93,26.75,12.23,0.97,22.10,...,NaN,13.24,87.09,1.68,220.89,0.00,0.00,50.85,992.45,NaN
364,2024-12-30,101.08,137.47,18.71,49.56,39.57,22.69,14.58,1.01,26.27,...,NaN,11.72,85.57,1.29,236.98,0.00,0.00,41.71,992.00,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         196.21        301.20       10.12        26.68   
1  2024-01-02         225.53        349.33       20.00        29.37   
2  2024-01-03         205.20        319.21       21.57        47.43   
3  2024-01-04         246.69        379.60       20.43        55.09   
4  2024-01-05         173.72        290.92       18.78        52.80   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      34.79        40.39         3.09        1.27          11.12   
1      47.37        38.98         3.54        1.29          12.26   
2      49.42        40.12         4.69        1.82          10.24   
3      45.92        36.88         7.32        1.74           3.71   
4      43.36        41.13         4.93        1.63          11.46   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.10             0.64    10.89   79.56      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.592879,1.082831,-0.495982,-1.120892,-0.236411,1.189199,-1.401612,0.011823,-1.788838,-0.693946,-1.002446,-1.726570,0.910200,-1.389617,-2.339370,0.0,0.0,-1.043512,1.354983
1,2024-01-02,2.037606,1.561002,0.504816,-1.003992,0.393556,1.013688,-1.344863,0.051161,-1.741083,-0.497705,-0.924156,-1.808912,0.781388,-1.443360,-1.786850,0.0,0.0,-0.890384,1.247092
2,2024-01-03,1.729239,1.261760,0.663850,-0.219155,0.496214,1.155590,-1.199840,1.093615,-1.825701,1.072218,0.240404,-1.854927,1.420202,-1.255259,-2.104921,0.0,0.0,-1.235929,1.229596
3,2024-01-04,2.358561,1.861734,0.548373,0.113727,0.320945,0.752289,-0.868177,0.936263,-2.099243,2.151539,1.581116,-1.806491,1.353174,0.222682,1.472440,0.0,0.0,-1.621945,1.270420
4,2024-01-05,1.251749,0.980699,0.381236,0.014210,0.192748,1.281311,-1.169574,0.719905,-1.774595,1.660939,1.385391,-1.705984,1.433608,-1.013414,-0.129541,0.0,0.0,-1.725609,1.271878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.663258,0.565218,-0.248821,0.771670,1.979992,1.899955,-0.271689,1.132953,-1.658560,0.875977,1.473467,-1.416574,1.342682,-0.287879,-1.251761,0.0,0.0,-2.099318,1.459959
362,2024-12-28,-0.004620,-0.709838,1.850018,0.166745,0.776144,0.573044,-0.196025,-0.165198,-1.846646,-0.203345,0.279549,-1.408098,1.540854,-0.932799,-0.112867,0.0,0.0,-1.972224,1.382685
363,2024-12-29,0.152673,-0.580385,-0.010777,-0.253052,-0.229400,-0.508650,-0.248990,-0.578246,-1.328885,-0.595825,-0.875225,-1.442004,1.349094,1.620008,0.432327,0.0,0.0,-1.095581,1.605758
364,2024-12-30,0.149943,-0.543825,0.374145,-0.126591,0.002957,-1.014021,0.047363,-0.499570,-1.154203,-0.497705,-0.992659,-1.626063,1.260499,0.572014,0.838823,0.0,0.0,-1.311902,1.540148


In [10]:
df.to_excel('DRKarni2024.xlsx', index=False)